# 22 · Your own project: from idea to a GitHub portfolio piece

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lukasquaderer-lgtm/Linear-regression/blob/master/notebooks/22_your_own_project.ipynb)

Everything so far had a dataset and a question chosen for you. A project you design yourself, on real data, written
up clearly and published on GitHub, is the strongest evidence you can show an employer that you can do the job.

This notebook gives you a **tested project template**, a menu of project ideas with data sources, a 4-week plan, the
rubric to judge your work, and a publishing checklist.

In [ ]:
# Setup: run this cell first (Shift + Enter). Works in Jupyter, VS Code and Google Colab.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # hide library deprecation notices

# Use the local data folder if it exists, otherwise read the CSVs straight from GitHub (e.g. in Colab)
DATA = next((p for p in ["../data/", "../../data/"] if Path(p).exists()), "https://raw.githubusercontent.com/lukasquaderer-lgtm/Linear-regression/master/data/")
pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (8, 4)

# Answer checker: after an exercise, check("01.1") tells you whether your answer is right
import sys
if Path("../checks.py").exists():
    sys.path.insert(0, "..")  # solution notebooks live one folder down
try:
    from checks import check, check_all
except ImportError:  # in Colab: fetch the checker from GitHub
    import urllib.request
    urllib.request.urlretrieve("https://raw.githubusercontent.com/lukasquaderer-lgtm/Linear-regression/master/notebooks/checks.py", "checks.py")
    from checks import check, check_all
print("Ready. Data folder:", DATA)

## 1. Get the project template
The template is a small Python package with tests, a backtester that cannot look ahead, and an example. The cell below copies it into a folder called `my_project`.

In [ ]:
import shutil, subprocess, sys
from pathlib import Path
tpl = Path("../project_template")
if not tpl.exists():                                   # in Colab: fetch it from GitHub
    if not Path("qma_repo").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lukasquaderer-lgtm/Linear-regression.git", "qma_repo"], check=True)
    tpl = Path("qma_repo/project_template")
work = Path("my_project")
if work.exists():
    shutil.rmtree(work)
shutil.copytree(tpl, work)
for p in sorted(work.rglob("*")):
    if p.is_file():
        print(p)

In [ ]:
run = lambda *cmd: print(subprocess.run(list(cmd), cwd=work, capture_output=True, text=True).stdout[-900:])
run(sys.executable, "-m", "pytest", "-q")          # all tests should pass
run(sys.executable, "run_example.py")               # reproduces an example backtest

Open `my_project/src/quantproject/backtest.py` and `my_project/tests/test_backtest.py`: one test proves the backtester never uses data from after the rebalance date. Keep tests like that as you build.

## 2. Choose a question
A good project question is **specific, decision-relevant and answerable with free data in about four weeks**.
Pick one of these or adapt it:

| # | Question | Free data | Techniques |
|---|---|---|---|
| 1 | Does low-volatility investing work in Swiss equities after costs? | Yahoo Finance (SMI stocks) | Portfolio construction, backtest, notebook 17 |
| 2 | How much does the CHF move when the SNB surprises? | SNB data portal (data.snb.ch), Yahoo FX | Event study, regression, time series |
| 3 | Can a factor model explain Swiss fund returns? | Fund NAVs (Yahoo), Kenneth French data library | Multiple regression, misspecification tests |
| 4 | A VaR model for a Swiss pension portfolio: which method passes the backtest? | Yahoo Finance ETFs | VaR/ES, Kupiec, EWMA, filtered HS |
| 5 | Do momentum signals work in European sector ETFs? | Yahoo Finance (STOXX sector ETFs) | Walk-forward ML, leakage control |
| 6 | What drives Swiss real-estate fund premiums to NAV? | SIX/Yahoo prices, published NAVs | Regression, panel data cleaning |
| 7 | Sentiment of SNB or ECB statements vs market moves | Central-bank websites, Yahoo | Text analytics, LLMs (notebook 16) |
| 8 | Is gold a better crisis hedge than CHF bonds for a Swiss investor? | Yahoo Finance | Correlation regimes, stress tests, drawdowns |

**Scope it down.** “Predict the stock market” is not a project; “Does 12-1 momentum in 20 SMI stocks beat equal
weight after 0.1% costs, 2010–2024?” is.

## 3. Starter: download real data
Runs in Colab (needs internet). Swap in the tickers your question needs.

In [ ]:
TICKERS = ["NESN.SW", "NOVN.SW", "ROG.SW", "UBSG.SW", "ZURN.SW", "ABBN.SW"]
try:
    try:
        import yfinance as yf
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance"], check=False)
        import yfinance as yf
    px = yf.download(TICKERS, start="2015-01-01", end="2024-12-31", auto_adjust=True, progress=False)["Close"]
    if px.dropna(how="all").empty:
        raise RuntimeError("no data returned")
    print(px.shape); print(px.tail(3).round(2))
    (work / "data").mkdir(exist_ok=True); px.to_csv(work / "data" / "prices.csv")
    print("saved my_project/data/prices.csv; record today's date in data/README.md")
except Exception as e:
    print("Download not possible here:", repr(e)[:120], "→ run this in Colab, or start from the course datasets.")

## 4. A four-week plan
| Week | Goal | Done when |
|---|---|---|
| 1 | Question, data, cleaning | Clean dataset saved; data-quality checks written as tests (notebook 20) |
| 2 | Baseline and main method | Simple baseline (e.g. equal weight, buy-and-hold) and your method, both in `src/`, tested |
| 3 | Honest evaluation | Out-of-sample or walk-forward results, costs, robustness (sub-periods, parameters), significance |
| 4 | Write-up and publish | README with question, one chart, one table, limitations; repository public on GitHub |

## 5. Rubric (score yourself, 100 points)
| Criterion | Points | Full marks when |
|---|---|---|
| Question | 10 | Specific, decision-relevant, answerable |
| Data | 15 | Sources documented, cleaning in code, biases (survivorship, look-ahead) addressed |
| Method | 15 | Appropriate technique, simple baseline included |
| Evaluation | 25 | Out-of-sample, costs, robustness checks, uncertainty stated |
| Code quality | 15 | Functions in `src/`, tests pass, reproducible from a fresh clone |
| Communication | 20 | README a non-specialist can follow; limitations honest; one clear chart |

## 6. Publishing checklist
- [ ] Repository is public, with a clear name (e.g. `swiss-low-vol-backtest`)
- [ ] README starts with the question and the answer in two sentences
- [ ] `python -m pytest -q` passes on a fresh clone; `requirements.txt` is complete
- [ ] No API keys, passwords or licensed data committed
- [ ] One chart and one results table in the README, with the period and costs stated
- [ ] A “Limitations” section that a sceptical reviewer would accept
- [ ] Link it on your CV and LinkedIn; be ready to explain one design decision and one thing you would do differently

**In an interview**, expect: “What would make this result wrong?” Answer with the biases you checked for
(look-ahead, survivorship, data snooping, costs), and you will stand out.